# Phase 1 Final Comparator Artifact Manifest / Readiness Plan

Notebook fix marker: `phase1_final_comparator_artifact_plan_v1_20260421`

Purpose: record the final comparator artifact schema required before any final Phase 1 comparator outputs can feed controls, calibration, influence or reporting. This notebook does **not** run final comparators, train models, compute final metrics, execute controls, or open headline claims.

Scientific integrity rule: final comparator artifacts must be produced by future final runners under the locked contract. Smoke metrics and smoke manifests cannot satisfy this plan.

## Technical Basis And Boundary

This notebook follows the repository contract:

- `docs/01_v55_doc_code_crosswalk.md`: notebooks only orchestrate CLI; durable scientific logic lives in `src/` and tests.
- `docs/02_colab_local_runbook.md`: final claim-package plan must be reviewed before any claim-bearing runner is implemented.
- `configs/phase1/final_claim_package.json`: requires final fold logs, subject-level metrics, logits, split/feature manifests, leakage audit and comparator completeness table.
- `configs/phase1/final_comparator_artifacts.json`: defines the final comparator artifact schema and leakage requirements.
- `src/phase1/final_comparator_artifacts.py`: records missing final comparator manifests; it does not produce comparator evidence.

Expected current result: all final comparator manifests are missing, `claim_ready=false`, and no smoke metrics are promoted.

In [ ]:
# Cell 1 - Runtime setup, Drive mount, clone/pull repo, and unit tests.
# This cell intentionally runs tests before writing comparator artifact plan outputs.

from pathlib import Path
import base64
import getpass
import subprocess

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752') if IN_COLAB else Path.cwd()

def run(cmd, cwd=None, check=True, redact_token=True):
    display = list(map(str, cmd))
    if redact_token:
        display = ['<redacted-token-command>' if 'http.extraHeader=Authorization:' in item else item for item in display]
    print('$', ' '.join(display))
    result = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def github_auth_args():
    if not IN_COLAB:
        return []
    token = getpass.getpass('Nhập GitHub token cho repo private; Enter nếu runtime đã có quyền: ')
    if not token:
        return []
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return ['-c', f'http.extraHeader={header}']

GIT_AUTH_ARGS = github_auth_args()
if not REPO_DIR.exists():
    run(['git', *GIT_AUTH_ARGS, 'clone', REPO_URL, str(REPO_DIR)])
else:
    print(f'Repo exists: {REPO_DIR}')

run(['git', *GIT_AUTH_ARGS, 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Stop here; do not write final comparator artifact plan from an unclean codebase.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])
print('Unit tests passed; continuing to final comparator artifact plan checks.')

In [ ]:
# Cell 2 - Explicit source paths.
# Use the reviewed final claim-package plan from notebook 16. Do not silently follow latest.txt for claim-affecting planning.

from pathlib import Path

PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
FINAL_CLAIM_PACKAGE_PLAN_RUN = DRIVE_ROOT / 'artifacts/phase1_final_claim_package_plan/20260421T074334327016Z'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_comparator_artifact_plan'
ARTIFACT_CONFIG = REPO_DIR / 'configs/phase1/final_comparator_artifacts.json'

EXPECTED_LOCKED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
EXPECTED_CLAIM_PACKAGE_STATUS = 'phase1_final_claim_package_plan_recorded'
EXPECTED_COMPARATORS = ['A2', 'A2b', 'A2c_CORAL', 'A2d_riemannian', 'A3_distillation', 'A4_privileged']

print('Prereg bundle:', PREREG_BUNDLE)
print('Final claim-package plan run:', FINAL_CLAIM_PACKAGE_PLAN_RUN)
print('Artifact config:', ARTIFACT_CONFIG)
print('Plan output root:', OUTPUT_ROOT)

In [ ]:
# Cell 3 - JSON/hash helpers used only for orchestration review.

import hashlib
import json
from pathlib import Path


def read_json(path: Path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding='utf-8'))


def sha256_file(path: Path) -> str:
    path = Path(path)
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def latest_run(root: Path) -> Path:
    pointer = Path(root) / 'latest.txt'
    if not pointer.exists():
        raise FileNotFoundError(pointer)
    return Path(pointer.read_text(encoding='utf-8').strip())

In [ ]:
# Cell 4 - Validate prereg identity and final claim-package boundary.

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_LOCKED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

claim_summary = read_json(FINAL_CLAIM_PACKAGE_PLAN_RUN / 'phase1_final_claim_package_plan_summary.json')
claim_state = read_json(FINAL_CLAIM_PACKAGE_PLAN_RUN / 'phase1_final_claim_state_plan.json')
claim_contract = read_json(FINAL_CLAIM_PACKAGE_PLAN_RUN / 'phase1_final_claim_package_contract.json')
print('Claim-package plan status:', claim_summary.get('status'))
print('Claim-package claim_ready:', claim_summary.get('claim_ready'))
print('Required final comparators:', claim_summary.get('required_final_comparators'))
assert claim_summary.get('status') == EXPECTED_CLAIM_PACKAGE_STATUS
assert claim_summary.get('claim_ready') is False
assert claim_summary.get('headline_phase1_claim_open') is False
assert claim_state.get('full_phase1_claim_bearing_run_allowed') is False
for item in EXPECTED_COMPARATORS:
    assert item in claim_summary.get('required_final_comparators', []), f'Missing comparator: {item}'
assert claim_contract.get('claim_opening_rules', {}).get('smoke_metrics_allowed_as_claim_evidence') is False

In [ ]:
# Cell 5 - Hash-link final comparator artifact source, config and notebook.

HASH_TARGETS = [
    'configs/phase1/final_comparator_artifacts.json',
    'configs/phase1/final_claim_package.json',
    'src/phase1/final_comparator_artifacts.py',
    'src/phase1/final_claim_package.py',
    'src/cli.py',
    'tests/unit/test_phase1_final_comparator_artifacts.py',
    'notebooks/17_colab_phase1_final_comparator_artifact_plan.ipynb',
]

hash_manifest = []
for rel in HASH_TARGETS:
    path = REPO_DIR / rel
    exists = path.exists()
    hash_manifest.append({'path': rel, 'exists': exists, 'sha256': sha256_file(path) if exists else None})
print(json.dumps(hash_manifest, indent=2))
assert all(item['exists'] for item in hash_manifest), 'All final comparator artifact source/config/notebook files must exist.'

## Expected Artifact Contract

The CLI command below writes a timestamped non-claim plan under `artifacts/phase1_final_comparator_artifact_plan/`:

- `phase1_final_comparator_artifact_plan_inputs.json`
- `phase1_final_comparator_artifact_plan_summary.json`
- `phase1_final_comparator_artifact_plan_report.md`
- `phase1_final_comparator_artifact_contract.json`
- `phase1_final_comparator_manifest_status.json`
- `phase1_final_comparator_missing_artifacts.json`
- `phase1_final_comparator_leakage_requirements.json`
- `phase1_final_comparator_claim_state.json`
- `phase1_final_comparator_implementation_plan.json`

Expected current interpretation: final comparator manifests are missing, smoke metrics are not promoted, and claims remain closed.

In [ ]:
# Cell 6 - Run the CLI-backed final comparator artifact plan.
# This command records schema/readiness only. It does not run final comparator models.

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_comparator_artifact_plan',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--claim-package-run', str(FINAL_CLAIM_PACKAGE_PLAN_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--artifact-config', 'configs/phase1/final_comparator_artifacts.json',
    '--claim-package-config', 'configs/phase1/final_claim_package.json',
]
run(cmd, cwd=REPO_DIR)
print('Final comparator artifact plan command completed. Review missing manifests before implementing final comparator runners.')

In [ ]:
# Cell 7 - Verify plan artifacts and closeout state.

run_dir = latest_run(OUTPUT_ROOT)
print('Latest final comparator artifact plan output:', run_dir)
required_files = [
    'phase1_final_comparator_artifact_plan_inputs.json',
    'phase1_final_comparator_artifact_plan_summary.json',
    'phase1_final_comparator_artifact_plan_report.md',
    'phase1_final_comparator_artifact_contract.json',
    'phase1_final_comparator_manifest_status.json',
    'phase1_final_comparator_missing_artifacts.json',
    'phase1_final_comparator_leakage_requirements.json',
    'phase1_final_comparator_claim_state.json',
    'phase1_final_comparator_implementation_plan.json',
]
for name in required_files:
    exists = (run_dir / name).exists()
    print(f'{name} exists = {exists}')
    assert exists, f'Missing required artifact: {name}'

summary = read_json(run_dir / 'phase1_final_comparator_artifact_plan_summary.json')
contract = read_json(run_dir / 'phase1_final_comparator_artifact_contract.json')
manifest_status = read_json(run_dir / 'phase1_final_comparator_manifest_status.json')
claim_state = read_json(run_dir / 'phase1_final_comparator_claim_state.json')
missing = read_json(run_dir / 'phase1_final_comparator_missing_artifacts.json')

print(json.dumps({
    'run_dir': str(run_dir),
    'summary_status': summary.get('status'),
    'artifact_plan_status': summary.get('artifact_plan_status'),
    'claim_ready': summary.get('claim_ready'),
    'headline_phase1_claim_open': summary.get('headline_phase1_claim_open'),
    'full_phase1_claim_bearing_run_allowed': summary.get('full_phase1_claim_bearing_run_allowed'),
    'required_final_comparators': summary.get('required_final_comparators'),
    'all_final_comparator_manifests_present': summary.get('all_final_comparator_manifests_present'),
    'smoke_metrics_promoted': summary.get('smoke_metrics_promoted'),
    'contract_matches_claim_package': contract.get('comparator_contract_matches_claim_package'),
    'artifact_schema_matches_claim_package': contract.get('artifact_schema_matches_claim_package'),
    'manifest_status': manifest_status.get('status'),
    'blockers': claim_state.get('blockers'),
}, indent=2))

assert summary.get('status') == 'phase1_final_comparator_artifact_plan_recorded'
assert summary.get('artifact_plan_status') == 'planning_non_claim'
assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert summary.get('full_phase1_claim_bearing_run_allowed') is False
assert summary.get('all_final_comparator_manifests_present') is False
assert summary.get('smoke_metrics_promoted') is False
assert contract.get('comparator_contract_matches_claim_package') is True
assert contract.get('artifact_schema_matches_claim_package') is True
assert manifest_status.get('claim_evaluable') is False
assert len(manifest_status.get('comparators', [])) == len(EXPECTED_COMPARATORS)
for row in manifest_status.get('comparators', []):
    assert row.get('status') == 'final_comparator_manifest_missing'
    assert row.get('claim_evaluable') is False
    assert row.get('smoke_metrics_promoted') is False
for expected_blocker in [
    'final_comparator_artifact_plan_not_locked',
    'final_comparator_artifact_manifests_missing',
    'final_comparator_completeness_table_missing',
    'final_comparator_outputs_not_claim_evaluable',
    'claim_blocked_until_final_comparator_artifacts_exist',
]:
    assert expected_blocker in claim_state.get('blockers', []), f'Missing blocker: {expected_blocker}'
print('\nNOT OK TO CLAIM: This final comparator artifact plan does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')

## Next Engineering Boundary

Allowed next implementation work: implement final comparator runners/manifests against this artifact contract, starting with split/feature manifests and leakage audit surfaces before writing claim-bearing metrics.

Not allowed: marking any final comparator claim-evaluable, promoting smoke metrics, or opening controls/calibration/influence/reporting before final comparator artifacts exist.